In [1]:
import sys
import json
from pathlib import Path

WOOSH_ROOT = Path("/workspace/Woosh")
PROJECT_ROOT = Path("/workspace/storyfx")

print("Woosh:", WOOSH_ROOT)
print("StoryFX:", PROJECT_ROOT)

Woosh: /workspace/Woosh
StoryFX: /workspace/storyfx


In [2]:
import torch
import torchaudio

from woosh.inference.flowmatching_sampler import flowmatching_integrate
from woosh.model.ldm import LatentDiffusionModel
from woosh.components.base import LoadConfig

In [3]:
%cd /workspace/Woosh

/workspace/Woosh


In [4]:
device = "cuda" if torch.cuda.is_available() else "cpu"

COMPONENT_PATH = (
    WOOSH_ROOT
    / "checkpoints"
    / "Woosh-Flow"
)

if not COMPONENT_PATH.exists():
    raise FileNotFoundError(
        f"Woosh-Flow checkpoint not found: {COMPONENT_PATH}"
    )

print("Device:", device)
print("Checkpoint:", COMPONENT_PATH)

ldm = LatentDiffusionModel(
    LoadConfig(
        path=str(COMPONENT_PATH)
    )
)

ldm = ldm.eval().to(device)

print("Woosh loaded!")

Device: cuda
Checkpoint: /workspace/Woosh/checkpoints/Woosh-Flow
Woosh loaded!


In [5]:
def generate_sfx(
    description,
    output_path,
    duration_s=5.0,
    cfg=4.5,
    atol=0.001,
    rtol=0.001,
):
    batch_size = 1

    latent_frames = int(duration_s * 100) + 1

    noise = torch.randn(
        batch_size,
        128,
        latent_frames,
        device=device
    )

    cond = ldm.get_cond(
        {
            "audio": None,
            "description": [description]
        },
        no_dropout=True,
        device=device,
    )

    with torch.inference_mode():
        x_fake, steps = flowmatching_integrate(
            ldm,
            noise=noise,
            cond=cond,
            cfg=cfg,
            atol=atol,
            rtol=rtol,
            return_steps=True,
            device=device,
            dtype=torch.float64,
        )

        audio = ldm.autoencoder.inverse(x_fake)

    audio = audio.cpu()

    max_abs = torch.max(torch.abs(audio[0]))
    normalization_factor = max_abs if max_abs > 1.0 else 1.0
    audio = audio[0] / normalization_factor

    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)

    torchaudio.save(
        str(output_path),
        audio,
        sample_rate=48000,
    )

    print(f"Generated in {steps + 1} integration steps")
    print(f"Saved to {output_path}")

In [6]:
def extract_excerpt_id(path, suffix):
    path = Path(path)

    filename = path.stem

    if filename.startswith("pred_"):
        filename = filename[len("pred_"):]

    if not filename.endswith(suffix):
        raise ValueError(
            f"Expected filename ending with '{suffix}', "
            f"got: {path.name}"
        )

    excerpt_id = filename[:-len(suffix)]

    return excerpt_id

In [7]:
def discrete_sfx_generation(schedule):
    schedule = Path(schedule)

    discrete_schedule = json.loads(
        schedule.read_text(
            encoding="utf-8"
        )
    )

    excerpt_id = extract_excerpt_id(
        schedule,
        "_discrete_sfx_schedule",
    )

    output_dir = (
        PROJECT_ROOT
        / "outputs"
        / "assets"
        / excerpt_id
        / "discrete_assets"
    )

    output_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    generated_assets = []

    print(
        f"Generating discrete SFX for "
        f"{excerpt_id}"
    )

    for sfx in discrete_schedule:
        sfx_id = sfx[
            "discrete_sfx_id"
        ]

        caption = sfx[
            "caption"
        ]

        duration_time = float(
            sfx["duration_time"]
        )

        output_path = (
            output_dir
            / f"{sfx_id}.wav"
        )

        print(
            f"\n{sfx_id}"
        )
        print(
            f"Caption: {caption}"
        )
        print(
            f"Duration: "
            f"{duration_time:.2f}s"
        )

        generate_sfx(
            description=caption,
            output_path=output_path,
            duration_s=duration_time,
        )

        generated_assets.append(
            output_path
        )

    print(
        f"\nGenerated "
        f"{len(generated_assets)} "
        f"discrete assets."
    )

    return generated_assets

In [8]:
DISCRETE_SCHEDULE = (
    PROJECT_ROOT
    / "outputs"
    / "schedules"
    / "excerpt_001_discrete_sfx_schedule.json"
)

discrete_assets = discrete_sfx_generation(
    DISCRETE_SCHEDULE
)

Generating discrete SFX for excerpt_001

discrete_sfx000
Caption: grandfather clock chiming
Duration: 2.00s


/workspace/Woosh/.venv/lib/python3.13/site-packages/torchaudio/_backend/utils.py:337: UserWarning: In 2.9, this function's implementation will be changed to use torchaudio.save_with_torchcodec` under the hood. Some parameters like format, encoding, bits_per_sample, buffer_size, and ``backend`` will be ignored. We recommend that you port your code to rely directly on TorchCodec's encoder instead: https://docs.pytorch.org/torchcodec/stable/generated/torchcodec.encoders.AudioEncoder
  warnings.warn(
/workspace/Woosh/.venv/lib/python3.13/site-packages/torchaudio/_backend/ffmpeg.py:247: UserWarning: torio.io._streaming_media_encoder.StreamingMediaEncoder has been deprecated. This deprecation is part of a large refactoring effort to transition TorchAudio into a maintenance phase. The decoding and encoding capabilities of PyTorch for both audio and video are being consolidated into TorchCodec. Please see https://github.com/pytorch/audio/issues/3902 for more information. It will be removed fro

Generated in 76 integration steps
Saved to /workspace/storyfx/outputs/assets/excerpt_001/discrete_assets/discrete_sfx000.wav

discrete_sfx001
Caption: heavy fur cloak rustling
Duration: 1.14s
Generated in 76 integration steps
Saved to /workspace/storyfx/outputs/assets/excerpt_001/discrete_assets/discrete_sfx001.wav

discrete_sfx002
Caption: metal lantern abruptly lifted
Duration: 1.49s
Generated in 58 integration steps
Saved to /workspace/storyfx/outputs/assets/excerpt_001/discrete_assets/discrete_sfx002.wav

discrete_sfx003
Caption: hurried footsteps
Duration: 2.80s
Generated in 82 integration steps
Saved to /workspace/storyfx/outputs/assets/excerpt_001/discrete_assets/discrete_sfx003.wav

discrete_sfx004
Caption: massive oak doors slowly creaking open
Duration: 3.26s
Generated in 76 integration steps
Saved to /workspace/storyfx/outputs/assets/excerpt_001/discrete_assets/discrete_sfx004.wav

discrete_sfx005
Caption: massive wooden doors booming shut
Duration: 0.94s
Generated in 76 int

In [9]:
def ambient_sfx_generation(specification):
    specification = Path(specification)

    ambient_spec = json.loads(
        specification.read_text(
            encoding="utf-8"
        )
    )

    excerpt_id = extract_excerpt_id(
        specification,
        "_ambient_sfx_specification",
    )

    output_dir = (
        PROJECT_ROOT
        / "outputs"
        / "assets"
        / excerpt_id
        / "ambient_assets"
    )

    output_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    AMBIENT_DURATION = 7.00
    generated_assets = []

    print(
        f"Generating ambient SFX for "
        f"{excerpt_id}"
    )

    for segment in ambient_spec:
        segment_id = segment[
            "segment_id"
        ]

        print(
            f"\n--- {segment_id} ---"
        )

        # ==========================================
        # Place
        # ==========================================

        place = segment.get(
            "place"
        )

        if place is not None:
            output_path = (
                output_dir
                / f"{segment_id}_place.wav"
            )

            print(
                f"Place: {place}"
            )

            generate_sfx(
                description=place,
                output_path=output_path,
                duration_s=AMBIENT_DURATION,
            )

            generated_assets.append(
                output_path
            )

        # ==========================================
        # Weather
        # ==========================================

        weather = segment.get(
            "weather"
        )

        if weather is not None:
            output_path = (
                output_dir
                / f"{segment_id}_weather.wav"
            )

            print(
                f"Weather: {weather}"
            )

            generate_sfx(
                description=weather,
                output_path=output_path,
                duration_s=AMBIENT_DURATION,
            )

            generated_assets.append(
                output_path
            )

        # ==========================================
        # Special
        # ==========================================

        special = segment.get(
            "special"
        )

        special_caption = None

        if isinstance(
            special,
            dict,
        ):
            special_caption = special.get(
                "caption"
            )

        if special_caption is not None:
            output_path = (
                output_dir
                / f"{segment_id}_special.wav"
            )

            print(
                f"Special: "
                f"{special_caption}"
            )

            generate_sfx(
                description=special_caption,
                output_path=output_path,
                duration_s=AMBIENT_DURATION,
            )

            generated_assets.append(
                output_path
            )

    print(
        f"\nGenerated "
        f"{len(generated_assets)} "
        f"ambient assets."
    )

    return generated_assets

In [10]:
AMBIENT_SPEC = (
    PROJECT_ROOT
    / "outputs"
    / "composition_plans"
    / "pred_excerpt_001_ambient_sfx_specification.json"
)

ambient_assets = ambient_sfx_generation(
    AMBIENT_SPEC
)

Generating ambient SFX for excerpt_001

--- segment_001 ---
Weather: windy rainstorm weather outside
Generated in 76 integration steps
Saved to /workspace/storyfx/outputs/assets/excerpt_001/ambient_assets/segment_001_weather.wav

--- segment_002 ---
Weather: windy rainstorm weather
Generated in 58 integration steps
Saved to /workspace/storyfx/outputs/assets/excerpt_001/ambient_assets/segment_002_weather.wav

--- segment_003 ---
Weather: muffled windy rain weather outside
Generated in 58 integration steps
Saved to /workspace/storyfx/outputs/assets/excerpt_001/ambient_assets/segment_003_weather.wav

--- segment_004 ---
Weather: muffled windy rain weather outside
Generated in 52 integration steps
Saved to /workspace/storyfx/outputs/assets/excerpt_001/ambient_assets/segment_004_weather.wav
Special: recurring supernatural whispers from walls
Generated in 70 integration steps
Saved to /workspace/storyfx/outputs/assets/excerpt_001/ambient_assets/segment_004_special.wav

--- segment_005 ---
We